In [0]:
from pyspark.sql import functions as F

df_posts = spark.table("bronze_posts_raw")

silver_posts = (
    df_posts
    .select(
        F.col("_Id").cast("int").alias("post_id"),
        F.col("_PostTypeId").cast("int").alias("post_type_id"),
        F.col("_OwnerUserId").cast("int").alias("user_id"),
        F.col("_ParentId").cast("int").alias("parent_post_id"),
        F.col("_AcceptedAnswerId").cast("int").alias("accepted_answer_id"),
        F.to_timestamp("_CreationDate").alias("creation_date"),
        F.col("_Score").cast("int").alias("score"),
        F.col("_ViewCount").cast("int").alias("view_count"),
        F.col("_AnswerCount").cast("int").alias("answer_count"),
        F.col("_CommentCount").cast("int").alias("comment_count"),
        F.col("_Tags").alias("tags")
    )
    .withColumn(
        "post_type",
        F.when(F.col("post_type_id") == 1, "question")
         .when(F.col("post_type_id") == 2, "answer")
    )
    .fillna({
        "view_count":0,
        "score":0,
        "answer_count":0,
        "comment_count":0
    })
)

silver_posts.write.format("delta").mode("overwrite").saveAsTable("silver.posts")

In [0]:
silver_questions = (
    spark.table("silver.posts")
    .filter(F.col("post_type") == "question")
    .select(
        "post_id",
        "user_id",
        "accepted_answer_id",
        "creation_date",
        "score",
        "view_count",
        "answer_count",
        "comment_count",
        "tags"
    )
)

silver_questions.write.format("delta").mode("overwrite").saveAsTable("silver.questions")

In [0]:
silver_answers = (
    spark.table("silver.posts")
    .filter(F.col("post_type") == "answer")
    .select(
        "post_id",
        "parent_post_id",
        "user_id",
        "creation_date",
        "score",
        "comment_count"
    )
)

silver_answers.write.format("delta").mode("overwrite").saveAsTable("silver.answers")

In [0]:
df_users = spark.table("bronze_users_raw")

silver_users = (
    df_users
    .select(
        F.col("_Id").cast("int").alias("user_id"),
        F.col("_DisplayName").alias("display_name"),
        F.col("_Reputation").cast("int").alias("reputation"),
        F.col("_Location").alias("location"),
        F.to_timestamp("_CreationDate").alias("creation_date")
    )
)

silver_users.write.format("delta").mode("overwrite").saveAsTable("silver.users")

In [0]:
df_comments = spark.table("bronze_comments_raw")

silver_comments = (
    df_comments
    .select(
        F.col("_Id").cast("int").alias("comment_id"),
        F.col("_PostId").cast("int").alias("post_id"),
        F.col("_UserId").cast("int").alias("user_id"),
        F.col("_Score").cast("int").alias("score"),
        F.to_timestamp("_CreationDate").alias("creation_date")
    )
)

silver_comments.write.format("delta").mode("overwrite").saveAsTable("silver.comments")

In [0]:
df_votes = spark.table("bronze_votes_raw")

silver_votes = (
    df_votes
    .select(
        F.col("_Id").cast("int").alias("vote_id"),
        F.col("_PostId").cast("int").alias("post_id"),
        F.col("_VoteTypeId").cast("int").alias("vote_type_id"),
        F.to_timestamp("_CreationDate").alias("creation_date")
    )
)

silver_votes.write.format("delta").mode("overwrite").saveAsTable("silver.votes")

In [0]:
post_tags = (
    spark.table("silver.questions")
    .filter(F.col("tags").isNotNull())
    .withColumn(
        "tag",
        F.explode(
            F.split(
                F.regexp_replace("tags", r"^\||\|$", ""),
                r"\|"
            )
        )
    )
    .select(
        F.col("post_id"),
        F.col("tag").alias("tag_name")
    )
)

post_tags.write.format("delta").mode("overwrite").saveAsTable("silver.post_tags")

In [0]:
df_tags = spark.table("bronze_tags_raw")

silver_tags = (
    df_tags
    .select(
        F.col("_TagName").alias("tag_name"),
        F.col("_Count").cast("int").alias("tag_count")
    )
)

silver_tags.write.format("delta").mode("overwrite").saveAsTable("silver.tags")